Dans ce fichier, nous allons tester le modèle PPO pour répondre à notre problématique.
Commençons par les pip install et les imports

In [ ]:
!pip install "stable-baselines3[extra]" wandb pandas plotly notebook ta sb3_contrib

In [ ]:
import numpy as np
import pandas as pd
import gymnasium as gym
import gym_trading_env
from sb3_contrib import RecurrentPPO
import wandb
from gym_trading_env.wrapper import DiscreteActionsWrapper
from wandb.integration.sb3 import WandbCallback
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.env_checker import check_env
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import ta 

In [ ]:
os.environ["WANDB_ERROR_REPORTING"] = "False" 
os.environ["WANDB_CONSOLE"] = "off"

Commençons par un prermier modèle simple, pour comprendre le projet. Nous testons avec les features de base que le projet nous donne.
Nous allons entrainer notre modèle sur 250 000 pas pour commencer
Après avoir été entrainer, tous les modèles seront testés sur 30 épisodes pour étudier leurs performances

In [ ]:
def preprocess(df):
    df = df.sort_index()
    df = df.dropna()
    df = df.drop_duplicates()

    df['feature_close'] = (df['close'] - df['close'].mean()) / df['close'].std()

    return df

env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

config1 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 250_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config1,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model1 = PPO("MlpPolicy",env,tensorboard_log=f"runs/{run.id}")
model1.learn(
    total_timesteps=config1["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model1.save("PPO_1")


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.env.name 
    
    while not done:
        action, _states = model1.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action.tolist())
        done = terminated or truncated

    if done:
        history = env.env.historical_info
        df_history = pd.DataFrame(history)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

Nous avons testé plusieurs nombre de pas de temps avec comme plus grand 250_000 mais la conclusion est que actuellement, les résultats sont trop aléatoirs, parfois l'agent apprend bien mais il n'est pas enclin à faire mieux que le marché. Il se contente de toujours garder son argent.
Nous allons améliorer notre programme de 2 manières:
    - Changer la fonction de reward, pour qu'elle récompense plus l'exploration
    - ajouter des features pour que l'agent comprenne mieux le marché
    

In [ ]:
#Fonction de preprocess

def preprocess_v2(df):
    df = df.sort_index()
    df = df.dropna()
    df = df.drop_duplicates()

    # Valeur des différentes features par rapport à la moyenne
    df['feature_close'] = (df['close'] - df['close'].mean()) / df['close'].std()
    df['feature_high'] = (df['high'] - df['high'].mean()) / df['high'].std()
    df['feature_low'] = (df['low'] - df['low'].mean()) / df['low'].std()

    #Valeur des ratios entre 2 des 4 niveaux (haut/bas ou fermeture/ouverture)
    df['feature_ratio_high_low'] = (df['high'] / df['low'])
    df['feature_ratio_close_open'] = (df['close'] / df['open'])
    df['feature_ratio_high_close'] = (df['high'] / df['close'])

    #Valeur des features par rapport à leur max
    df['feature_max_ratio_close'] = (df['close'] - df['close'].max()) / df['close'].max()
    df['feature_max_ratio_open'] = (df['open'] - df['open'].max()) / df['open'].max()
    df['feature_max_ratio_low'] = (df['low'] - df['low'].max()) / df['low'].max()
    df['feature_max_ratio_high'] = (df['high'] - df['high'].max()) / df['high'].max()

    #Valeur des ratios par rapport à leur max entre 2 des 4 niveaux (haut/bas ou fermeture/ouverture)
    df['feature_ratio_max_high_low'] = (df['feature_ratio_high_low'] - df['feature_ratio_high_low'].max()) / df['feature_ratio_high_low'].max()
    df['feature_ratio_max__close_open'] = (df['feature_ratio_close_open'] - df['feature_ratio_close_open'].max()) / df['feature_ratio_close_open'].max()
    df['feature_ratio_max_high_close'] = (df['feature_ratio_high_close'] - df['feature_ratio_high_close'].max()) / df['feature_ratio_high_close'].max()

    return df

#Fonction de récompenses

def metric_portfolio_valuation(history):
    return round(history['portfolio_valuation', -1], 2)

def reward_function(history):
    return history['step']*(history['portfolio_valuation', -1] - history['portfolio_valuation', -2])

def reward_function_rapport_market(history):
    return 10*((history['portfolio_valuation', -1] / history['portfolio_valuation', -2]) - (history['data_close', -1] / history['data_open', -1]))

def reward_function_rapport_market_v2(history):
    weight = 0 
    if (history['position',1] == history['position',-2]):
        weight = 0.1
    return 10*(history['data_open', -2] / history['data_close', -1])*((history['portfolio_valuation', -1] / history['portfolio_valuation', -2]) - (history['data_close', -1] / history['data_open', -1])) - weight 


On teste la fonction de reward : reward_function qui fait attention à l'étape (step) de l'épisode, plus l'épisode est avancé, plus l'agent est puni pour perdre de l'argent

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess_v2,
    reward_function=reward_function,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)
env = DiscreteActionsWrapper(env, positions=[-1,-0.8,-0.6,-0.4,-0.2,0,0.2,0.4,0.5,0.6,0.8,1,1.2,1.4,1.6,1.8,2])
obs, _ = env.reset()

config2 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 300_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config2,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model2 = PPO("MlpPolicy",env,tensorboard_log=f"runs/{run.id}")
model2.learn(
    total_timesteps=config2["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model2.save("PPO_2")


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.env.name 
    
    while not done:
        action, _states = model2.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action.tolist())
        done = terminated or truncated

    if done:
        history = env.env.historical_info
        df_history = pd.DataFrame(history)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

On teste la fonction reward_function_rapport_market. Cette fonction compare l'argent gagné/perdu par l'agent par rapport au marché

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess_v2,
    reward_function=reward_function_rapport_market,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)
env = DiscreteActionsWrapper(env, positions=[-1,-0.8,-0.6,-0.4,-0.2,0,0.2,0.4,0.5,0.6,0.8,1,1.2,1.4,1.6,1.8,2])
obs, _ = env.reset()

config2 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 300_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config2,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model3 = PPO("MlpPolicy",env,tensorboard_log=f"runs/{run.id}")
model3.learn(
    total_timesteps=config2["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model3.save("PPO_3")


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.env.name 
    
    while not done:
        action, _states = model3.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action.tolist())
        done = terminated or truncated

    if done:
        history = env.env.historical_info
        df_history = pd.DataFrame(history)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

On teste la fonction reward_function_rapport_market_v2. Cette fonction fait le rapport au lieu de faire la différence et ajoute un "poids" en fonction de si l'agent reste sur la même position qu'avant ou non. Pour le forcer à faire des ventes.

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess_v2,
    reward_function=reward_function_rapport_market_v2,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)
env = DiscreteActionsWrapper(env, positions=[-1,-0.8,-0.6,-0.4,-0.2,0,0.2,0.4,0.5,0.6,0.8,1,1.2,1.4,1.6,1.8,2])
obs, _ = env.reset()

config2 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 300_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config2,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model4 = PPO("MlpPolicy",env,tensorboard_log=f"runs/{run.id}")
model4.learn(
    total_timesteps=config2["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model4.save("PPO_4")


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.env.name 
    
    while not done:
        action, _states = model4.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action.tolist())
        done = terminated or truncated

    if done:
        history = env.env.historical_info
        df_history = pd.DataFrame(history)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

Nos résultats s'améliorent mais ne sont toujours pas satisfaisant. Nous allons utiliser la librairie pandas_ta pour avoir des features de finance et ainsi pouvoir avoir de meilleurs résultats. Nous allons augmenter également le temps d'entrainement en passant de 300_000 steps à 1_000_000. ainsi que 2 nouvelles fonctions de récompenses qui prend le log du rapport de l'agent gagné par l'agent et l'autre juste le rapport de l'argent gagné

In [ ]:
def preprocess_v3(df):
    df = df.copy()
    
    df['feature_log_ret'] = np.log(df['close'] / df['close'].shift(1))

    df['feature_rsi'] = ta.rsi(df['close'], length=14) / 100.0 
    
    macd = ta.macd(df['close'])
    df['feature_macd_norm'] = macd['MACD_12_26_9'] / df['close']
    
    df['feature_atr_norm'] = ta.atr(df['high'], df['low'], df['close'], length=14) / df['close']
    
    df['feature_hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
    df['feature_hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)

    df = df.dropna()
    
    df = df.replace([np.inf, -np.inf], 0)

    return df

def reward_log_returns(history):
    prev_val = history['portfolio_valuation', -2]
    curr_val = history['portfolio_valuation', -1]
    
    if prev_val == 0: return 0
    
    reward = np.log(curr_val / prev_val)
    return reward

def reward_returns(history):
    prev_val = history['portfolio_valuation', -2]
    curr_val = history['portfolio_valuation', -1]
    
    if prev_val == 0: return 0
    
    reward = (curr_val - prev_val) / prev_val
    return reward


On teste la fonction de reward : reward_log_returns

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess_v3,
    reward_function=reward_log_returns,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)
env = DiscreteActionsWrapper(env, positions=[-1, 0, 1])
obs, _ = env.reset()

config3 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_000_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config3,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model5 = PPO("MlpPolicy",env,tensorboard_log=f"runs/{run.id}")
model5.learn(
    total_timesteps=config2["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model5.save("PPO_5")


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.env.name 
    
    while not done:
        action, _states = model5.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action.tolist())
        done = terminated or truncated

    if done:
        history = env.env.historical_info
        df_history = pd.DataFrame(history)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

On teste la fonction de reward : reward_returns

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess_v3,
    reward_function=reward_returns,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)
env = DiscreteActionsWrapper(env, positions=[-1, 0, 1])
obs, _ = env.reset()

config3 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_000_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config3,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model6 = PPO("MlpPolicy",env,tensorboard_log=f"runs/{run.id}")
model6.learn(
    total_timesteps=config2["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model6.save("PPO_6")


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.env.name 
    
    while not done:
        action, _states = model6.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action.tolist())
        done = terminated or truncated

    if done:
        history = env.env.historical_info
        df_history = pd.DataFrame(history)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

Notre meilleur modèle est le dernier, nous allons tenter de le tester en augmenter le nombre de positions disponible et en augmentant le nombre de total_timesteps d'entrainement. 

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess_v3,
    reward_function=reward_returns,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)
env = DiscreteActionsWrapper(env, positions=[-0.5,0,0.5,1,1.5])
obs, _ = env.reset()

config3 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config3,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model7 = PPO("MlpPolicy",env,tensorboard_log=f"runs/{run.id}")
model7.learn(
    total_timesteps=config2["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model7.save("PPO_7")


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.env.name 
    
    while not done:
        action, _states = model7.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action.tolist())
        done = terminated or truncated

    if done:
        history = env.env.historical_info
        df_history = pd.DataFrame(history)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

Pour améliorer notre modèle, nous allons modifier les hyperparamètres de notre modèle : 
- gamme : 0.995 pour augmenter le discount factor
- learning rate : une fonction décroissante au fur et à mesure de l'entrainement
- ent_coef : 0.01 taux d'exploration 

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess_v3,
    reward_function=reward_returns,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)
env = DiscreteActionsWrapper(env, positions=[-0.5,0,0.5,1,1.5])
obs, _ = env.reset()

config3 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config3,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model8 = PPO("MlpPolicy",env,tensorboard_log=f"runs/{run.id}",             gamma=0.995, 
            gae_lambda=0.95,
            learning_rate=linear_schedule(0.0003), 
            ent_coef=0.01, 
            batch_size=64,
            n_steps=2048,
            verbose=1)
model8.learn(
    total_timesteps=config2["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model8.save("PPO_8")


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.env.name 
    
    while not done:
        action, _states = model8.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action.tolist())
        done = terminated or truncated

    if done:
        history = env.env.historical_info
        df_history = pd.DataFrame(history)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

On teste une nouvelle fonction de récompense

In [ ]:
def reward_risk_adjusted(history):
    prev_val = history['portfolio_valuation', -2]
    curr_val = history['portfolio_valuation', -1]
    if prev_val == 0: return 0    
    log_return = np.log(curr_val / prev_val)
    risk_penalty = 0
    if log_return < 0:
        risk_penalty = abs(log_return) * 0.5 
        
    return log_return - risk_penalty

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess_v3,
    reward_function=reward_risk_adjusted,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)
env = DiscreteActionsWrapper(env, positions=[-0.5,0,0.5,1,1.5])
obs, _ = env.reset()

config3 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config3,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model9 = PPO("MlpPolicy",env,tensorboard_log=f"runs/{run.id}",             gamma=0.995, 
            gae_lambda=0.95,
            learning_rate=linear_schedule(0.0003), 
            ent_coef=0.01, 
            batch_size=64,
            n_steps=2048,
            verbose=1)
model9.learn(
    total_timesteps=config2["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model9.save("PPO_9")


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.env.name 
    
    while not done:
        action, _states = model9.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action.tolist())
        done = terminated or truncated

    if done:
        history = env.env.historical_info
        df_history = pd.DataFrame(history)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

Pour améliorer encore notre modèle, nous allons tester le modèle ReccurentPPO pour ajouter une couche LSTM pour que notre modèle se rappelle mieux des différentes expériences. Ainsi que des nouvelles features

In [ ]:
def preprocess_v4(df):
    df = df.copy()
    
    df['feature_log_ret'] = np.log(df['close'] / df['close'].shift(1))

    df['feature_rsi'] = ta.rsi(df['close'], length=14) / 100.0 
    
    macd = ta.macd(df['close'])
    df['feature_macd_norm'] = macd['MACD_12_26_9'] / df['close']
    for i in range(1, 6):
        df[f'feature_log_ret_lag_{i}'] = df['feature_log_ret'].shift(i)

    df['feature_atr_norm'] = ta.atr(df['high'], df['low'], df['close'], length=14) / df['close']
    
    df['feature_hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
    df['feature_hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)

    df = df.dropna().replace([np.inf, -np.inf], 0)
    return df

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess_v3,
    reward_function=reward_risk_adjusted,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)
env = DiscreteActionsWrapper(env, positions=[-0.5,0,0.5,1,1.5])
obs, _ = env.reset()

config4 = {
    "policy_type": "MlpLstmPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config3,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model10 = RecurrentPPO("MlpLstmPolicy",env,tensorboard_log=f"runs/{run.id}",
            gamma=0.995, 
            gae_lambda=0.95,
            learning_rate=linear_schedule(0.0003), 
            ent_coef=0.01, 
            batch_size=64,
            n_steps=2048,
            verbose=1)
            
model10.learn(
    total_timesteps=config2["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model10.save("PPO_10")


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.env.name 
    
    while not done:
        action, _states = model10.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action.tolist())
        done = terminated or truncated

    if done:
        history = env.env.historical_info
        df_history = pd.DataFrame(history)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

On va tester ce modèle avec une autre fonction de reward : reward_returns

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess_v3,
    reward_function=reward_returns,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)
env = DiscreteActionsWrapper(env, positions=[-0.5,0,0.5,1,1.5])
obs, _ = env.reset()

config4 = {
    "policy_type": "MlpLstmPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config3,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model11 = RecurrentPPO("MlpLstmPolicy",env,tensorboard_log=f"runs/{run.id}",
            gamma=0.995, 
            gae_lambda=0.95,
            learning_rate=linear_schedule(0.0003), 
            ent_coef=0.01, 
            batch_size=64,
            n_steps=2048,
            verbose=1)
            
model11.learn(
    total_timesteps=config2["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model11.save("PPO_11")


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.env.name 
    
    while not done:
        action, _states = model11.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action.tolist())
        done = terminated or truncated

    if done:
        history = env.env.historical_info
        df_history = pd.DataFrame(history)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

Nous remarquons que nos modèles y arrivent bien au début mais perdent beaucoup vers les fins d'épisodes pour régler ce problème nous allons augmenter ent_coef pour permettre à l'algorithme de mieux explorer.

nb : nous repassons sur un modèle PPO car il est plus rapide pour apprendre.

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess_v3,
    reward_function=reward_returns,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)
env = DiscreteActionsWrapper(env, positions=[-0.5,0,0.5,1,1.5])
obs, _ = env.reset()

config3 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config3,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model12 = PPO("MlpPolicy",env,tensorboard_log=f"runs/{run.id}",
            gamma=0.995, 
            gae_lambda=0.95,
            learning_rate=linear_schedule(0.0003), 
            ent_coef=0.1, 
            batch_size=64,
            n_steps=2048,
            verbose=1)
            
model12.learn(
    total_timesteps=config2["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model12.save("PPO_12")


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.env.name 
    
    while not done:
        action, _states = model12.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action.tolist())
        done = terminated or truncated

    if done:
        history = env.env.historical_info
        df_history = pd.DataFrame(history)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()

Nous avons beaucoup trop augmenter ent_coef. Il faudrait faire une fonction qui diminue en fonction de l'avancée de l'entrainenement. Donc utilisé notre fonction linear_schedule. 

In [ ]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess_v3,
    reward_function=reward_returns,
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)
env = DiscreteActionsWrapper(env, positions=[-0.5,0,0.5,1,1.5])
obs, _ = env.reset()

config3 = {
    "policy_type": "MlpPolicy",
    "initial_portfolio": 1_000,
    "total_timesteps" : 1_500_000,
    "env_id": "MultiDatasetTradingEnv"
}

run = wandb.init(
    project="rl-trading-project_billet_crespin_truong", 
    config=config3,
    sync_tensorboard=True,
    monitor_gym=False,
    save_code=False,
)

model13 = PPO("MlpPolicy",env,tensorboard_log=f"runs/{run.id}",
            gamma=0.995, 
            gae_lambda=0.95,
            learning_rate=linear_schedule(0.0003), 
            ent_coef=linear_schedule(0.05), 
            batch_size=64,
            n_steps=2048,
            verbose=1)
            
model13.learn(
    total_timesteps=config2["total_timesteps"],
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
)
model13.save("PPO_13")


In [ ]:
nb_episodes = 30

for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    done = False
    
    dataset_name = env.env.name 
    
    while not done:
        action, _states = model13.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action.tolist())
        done = terminated or truncated

    if done:
        history = env.env.historical_info
        df_history = pd.DataFrame(history)
        
        initial_val = df_history['portfolio_valuation'].iloc[0]
        final_val = df_history['portfolio_valuation'].iloc[-1]
        c_r_pourcent = ((final_val - initial_val) / initial_val) * 100
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])

        fig.add_trace(
            go.Scatter(y=df_history['data_close'], name="Prix (Close)", line=dict(color='gray', width=1)),
            secondary_y=False
        )

        fig.add_trace(
            go.Scatter(y=df_history['portfolio_valuation'], name="Portefeuille", line=dict(color='blue', width=2)),
            secondary_y=True
        )


        fig.update_layout(
            title=f"Episode {episode} : {dataset_name} (Compte de résultat: {c_r_pourcent:.2f}%)",
            xaxis_title="Steps"
        )

        wandb.log({
            "eval/episode": episode,
            "eval/dataset": dataset_name,
            "eval/cr_pourcent": c_r_pourcent,
            "eval/final_value": final_val,
            "eval/chart": wandb.Plotly(fig) 
        })
        
        print(f"Episode {episode} loggé : {dataset_name} -> PnL: {c_r_pourcent:.2f}%")

wandb.finish()